<a href="https://colab.research.google.com/github/camilavazquezz/colab/blob/main/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import kagglehub
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [27]:
# Data source: Kaggle - Telco Customer Churn
# IBM Sample Data Sets
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Dataset downloaded to:", path)
print("Files:", os.listdir(path))

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Dataset downloaded to: /kaggle/input/telco-customer-churn
Files: ['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [28]:
csv_path = os.path.join(
    path,
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(csv_path)

print("Number of customers:", len(df))
print("\nColumns:")
print(df.columns.tolist())

df.head()

Number of customers: 7043

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [29]:
# Select customer features
X = df[
    [
        'tenure',
        'MonthlyCharges',
        'SeniorCitizen',
        'Contract',
        'InternetService',
        'PaymentMethod',
        'TechSupport'
    ]
].copy()

# Convert Churn into numbers:
# No = 0 (customer stayed)
# Yes = 1 (customer churned)
y = df['Churn'].map({
    'No': 0,
    'Yes': 1
})

print("Customer Features:")
print(X.head())

print("\nChurn Target:")
print(y.head())

Customer Features:
   tenure  MonthlyCharges  SeniorCitizen        Contract InternetService  \
0       1           29.85              0  Month-to-month             DSL   
1      34           56.95              0        One year             DSL   
2       2           53.85              0  Month-to-month             DSL   
3      45           42.30              0        One year             DSL   
4       2           70.70              0  Month-to-month     Fiber optic   

               PaymentMethod TechSupport  
0           Electronic check          No  
1               Mailed check          No  
2               Mailed check          No  
3  Bank transfer (automatic)         Yes  
4           Electronic check          No  

Churn Target:
0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64


In [30]:
# Define numerical and categorical features
numerical_features = [
    'tenure',
    'MonthlyCharges',
    'SeniorCitizen'
]

categorical_features = [
    'Contract',
    'InternetService',
    'PaymentMethod',
    'TechSupport'
]

# Preprocess the data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [31]:
# Create a pipeline with preprocessing and Logistic Regression
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train the Logistic Regression model
model.fit(X_train, y_train)

print("Model trained successfully!")
print("Training customers:", len(X_train))
print("Testing customers:", len(X_test))

Model trained successfully!
Training customers: 5634
Testing customers: 1409


In [32]:
# Create a new customer to predict
new_customer = pd.DataFrame({
    'tenure': [6],
    'MonthlyCharges': [85.00],
    'SeniorCitizen': [0],
    'Contract': ['Month-to-month'],
    'InternetService': ['Fiber optic'],
    'PaymentMethod': ['Electronic check'],
    'TechSupport': ['No']
})

# Predict probability of churn
churn_probability = model.predict_proba(new_customer)[0][1]

# Set classification threshold
threshold = 0.5

# Classify customer based on the threshold
churn_prediction = 1 if churn_probability > threshold else 0

if churn_prediction == 1:
    risk_status = "At risk of churning"
else:
    risk_status = "Not at risk of churning"

print(f"Churn Probability: {churn_probability:.2%}")
print(f"Churn Prediction: {churn_prediction}")
print(f"Customer Status: {risk_status}")

Churn Probability: 69.49%
Churn Prediction: 1
Customer Status: At risk of churning


In [33]:
# Get feature names after preprocessing
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = model.named_steps['classifier'].coef_[0]

print("Model Coefficients:")

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.3f}")

Model Coefficients:
num__tenure: -0.748
num__MonthlyCharges: 0.234
num__SeniorCitizen: 0.098
cat__Contract_Month-to-month: 0.739
cat__Contract_One year: 0.027
cat__Contract_Two year: -0.769
cat__InternetService_DSL: -0.196
cat__InternetService_Fiber optic: 0.506
cat__InternetService_No: -0.313
cat__PaymentMethod_Bank transfer (automatic): -0.078
cat__PaymentMethod_Credit card (automatic): -0.147
cat__PaymentMethod_Electronic check: 0.341
cat__PaymentMethod_Mailed check: -0.119
cat__TechSupport_No: 0.345
cat__TechSupport_No internet service: -0.313
cat__TechSupport_Yes: -0.035
